# Text queries: BM25, Nomic and CLIP

**Question:** Which information does each retrieval signal preserve or miss?
We test color/shape, compound product names, functional listing claims, exact
terms, paraphrases and a negation stress case. CLIP has a text encoder: being
given text does not by itself make it the wrong method.

Cases were fixed before inspecting rankings. No router calls, model tuning,
production ranking changes, dataset writes or vector rebuilding occur here.
Run all cells in order, or run `python scripts/run_retrieval_experiments.py`
from the repository root using the project environment.


Evaluation history: queries and seeds stayed fixed. After a preliminary run, narrowed an overbroad title rule that rejected valid T-shirt dresses and added missing title-family controls to the bounded pool. This is exploratory evaluation, without a held-out test set.


**Observed index confound:** three of six fixed stored text vectors did not reproduce the current title/description/features input with the same model digest. Full-catalog comparisons therefore describe this index snapshot, not isolated encoder quality. The manifest records the sample check. Notebook 02 adds freshly encoded full text on the same small candidate pool as its title/description controls. No database vectors are overwritten. The cause of the mismatch is not established; input/preprocessing/provider behavior remain unresolved.


## 1. Read one common candidate universe

Every method searches the same apparel products with both ready vector types.
The manifest records model revisions, source identity, vector dimensions,
package versions and a hash of the actual texts/vectors. Fail on mixed revisions.

These are **component experiments**: the production router, phrase/title gates,
fallback and relevance cutoffs are deliberately excluded. The RRF reference uses
the application's unchanged k=60, equal 0.50 base weights and top-50 component pools.
Raw BM25 and cosine magnitudes are not comparable across methods.


In [1]:
from pathlib import Path
import json
import sys
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Run from the repository root or notebooks/"
sys.path.insert(0, str(ROOT))
from notebooks.retrieval_lab import (
    RESULTS, load_catalog, embeddings, centroid, ranked, exclude_examples,
    fuse_components, evidence, candidate_coverage, audit_text_sample, visual_review_summary, summaries, top_results, save_gallery, save_json, print_metrics,
)
from rag.bm25 import rank as bm25_rank

cases = json.loads((ROOT / "notebooks/cases.json").read_text(encoding="utf-8"))
products, text_vectors, image_vectors, manifest = load_catalog()
sample_ids = [seed["id"] for group in cases["seed_groups"] for seed in group["seeds"]]
manifest["text_vector_sample"] = audit_text_sample(products, text_vectors, sample_ids, manifest)
save_json("vector_integrity.json", {"manifest": manifest, "sample": manifest["text_vector_sample"]})
print(json.dumps(manifest, indent=2))


Stored/current-format text-vector sample: {"seed_ids": ["amazon-clothing-personal-pilot:B002JVWOO8", "amazon-clothing-personal-pilot:B0042ANDY4", "amazon-clothing-personal-pilot:B007UXNDX6", "amazon-clothing-personal-pilot:B002LZ84K0", "amazon-clothing-personal-pilot:B00DHHTAVS", "amazon-clothing-personal-pilot:B00DYMJLT2"], "cosines": [0.9999997791583279, 0.8995660951645259, 0.7263584627276618, 0.9999996368015095, 0.9999997154517701, 0.37431105030990075], "match_tolerance": 0.999, "matching": 3, "sample_count": 6, "all_match": false, "input_characters": [1075, 4484, 2235, 1764, 1386, 4866], "input_sha256": ["d16c606c29b2c9b0dad84b1224bf1a925033ba97cb23550df555175039191a0f", "30da446076d9bbbbeb335eaf99fdbd9e211ed9ec59efa938759aeb1b8ee06328", "698e52a2cf2581ded502c90b305019b6866be53529cb5a7810f1a7b402e03fcb", "a005ec33fc744e176f4c26f86c170b81debb706f792b4602a679dc48742a46ac", "036fcc008deb1574e118bf400523ca924e68b6d2494fe63dbe07e1bfe781016e", "df774c5a563749f30e6ed9e0581bb62de00a8fd3ef1

## 2. Encode the same ten requests

Nomic uses the same `search_query:` prefix as the application. CLIP compares
its text features with stored image features. Exact experiment inputs/vectors
are cached by immutable model revision under `.data/experiments/retrieval/`.
Cached reruns need no new inference for unchanged inputs. No generative LLM is used.


In [2]:
text_cases = cases["text_cases"]
queries = [case["query"] for case in text_cases]
query_text = embeddings(["search_query: " + query for query in queries], "text", manifest)
query_images = embeddings(queries, "image", manifest)
documents = [" ".join([p["title"] or "", p["description"] or "", *(p["features"] or [])])
             for p in products]
print("Cases:", queries)


Cases: ['blue dress', 'pink bow tie', 'black ankle boots', 'white low top sneakers', 'waterproof hiking boots', 'machine washable cotton shirt', 'sterling silver necklace with a lobster clasp', 'Ross-Simons pearl earrings', 'something soft to wrap around my neck on a cold day', 'dress that is not blue']


## 3. Rank, inspect and save

`family@5` checks a published regex against titles. `listing@5` also requires
literal request evidence in title/description/features/brand. Both use a fixed
denominator of five; neither is human-judged relevance or visual accuracy.
Seller copy can describe multiple variants and can disagree with the photo.
Negation is not automatically scored: missing the word blue cannot prove not blue.

Use the galleries to judge visual fit separately. Keep poor and surprising cases;
do not select queries afterwards to manufacture a winner.


In [3]:
all_metrics, results = [], {}
for number, case in enumerate(text_cases):
    methods = {
        "bm25": bm25_rank(case["query"], documents),
        "nomic_text": ranked(text_vectors @ query_text[number], products),
        "clip_text": ranked(image_vectors @ query_images[number], products),
    }
    methods["rrf_reference"] = fuse_components(methods, products)
    rows = summaries(methods, products, case)
    all_metrics.extend(rows)
    gallery = save_gallery(methods, products, "text_" + case["id"])
    results[case["id"]] = {"query": case["query"], "kind": case["kind"], "gallery": gallery,
                           "top5": top_results(methods, products)}
print_metrics(all_metrics)
save_json("text_queries.json", {"manifest": manifest, "cases": text_cases,
          "candidate_evidence_counts": candidate_coverage(products, text_cases),
          "metrics": all_metrics, "results": results})
print("Saved:", RESULTS / "text_queries.json")


case                    method                      family@5  listing@5
blue_dress              bm25                            0.80       0.80
blue_dress              nomic_text                      1.00       0.60
blue_dress              clip_text                       0.40       0.00
blue_dress              rrf_reference                   1.00       0.60
pink_bow_tie            bm25                            0.60       0.00
pink_bow_tie            nomic_text                      0.80       0.00
pink_bow_tie            clip_text                       0.40       0.00
pink_bow_tie            rrf_reference                   0.60       0.00
black_ankle_boots       bm25                            0.80       0.20
black_ankle_boots       nomic_text                      0.80       0.40
black_ankle_boots       clip_text                       0.80       0.00
black_ankle_boots       rrf_reference                   1.00       0.40
white_sneakers          bm25                            0.80    

## 4. Visual review

Every row is a method; every column is its rank 1–5. Scores are shown only
for tracing a result, not as calibrated probabilities or cross-method scores.


### Blue dress

![Blue dress retrieval results](../observations/experiments/retrieval/figures/text_blue_dress.jpg)


### Pink bow tie

![Pink bow tie retrieval results](../observations/experiments/retrieval/figures/text_pink_bow_tie.jpg)


### Black ankle boots

![Black ankle boots retrieval results](../observations/experiments/retrieval/figures/text_black_ankle_boots.jpg)


### White low-top sneakers

![White low-top sneakers retrieval results](../observations/experiments/retrieval/figures/text_white_sneakers.jpg)


### Waterproof hiking boots

![Waterproof hiking boots retrieval results](../observations/experiments/retrieval/figures/text_waterproof_boots.jpg)


### Machine-washable cotton shirt

![Machine-washable cotton shirt retrieval results](../observations/experiments/retrieval/figures/text_cotton_shirt.jpg)


### Silver necklace and clasp

![Silver necklace and clasp retrieval results](../observations/experiments/retrieval/figures/text_silver_clasp.jpg)


### Brand and pearl earrings

![Brand and pearl earrings retrieval results](../observations/experiments/retrieval/figures/text_brand_earrings.jpg)


### A soft wrap for a cold day

![A soft wrap for a cold day retrieval results](../observations/experiments/retrieval/figures/text_warm_neck.jpg)


### Negation: not blue

![Negation: not blue retrieval results](../observations/experiments/retrieval/figures/text_exclude_blue.jpg)


## Interpretation and limits

Review `observations/experiments/retrieval/findings.md` for the actual run's
observations. Empty cells/absent figures mean the experiment has not run.
These convenience cases do not establish population accuracy, Recall@k,
statistical significance or production end-to-end quality. No independent
human relevance labels are available. Visual review is one analyst's inspection.

Background: [official CLIP code and normalized similarity example](https://github.com/openai/CLIP),
[Nomic task prefixes](https://huggingface.co/nomic-ai/nomic-embed-text-v1.5#task-instruction-prefixes),
[installed Ollama model family](https://ollama.com/library/nomic-embed-text).
The run manifest's digests identify the actual local weights.


## One explicit visual check: white low-top shoes

The 16 unique photos in this complete four-method grid were reviewed after retrieval, without blinding. The rubric accepts predominantly white/off-white uppers with small contrasting trim, and requires a low-top silhouette. This is an exploratory example, not held-out Precision@5. Labels and notes are editable in visual_judgments.json. New/ambiguous candidates remain unknown, never silently negative.


In [4]:
review = json.loads((ROOT / "notebooks/visual_judgments.json").read_text(encoding="utf-8"))
assert review["catalog_sha256"] == manifest["catalog_sha256"], "Photo judgments belong to another catalog snapshot"
visual_summary = visual_review_summary(results["white_sneakers"]["top5"], review)
print(json.dumps(visual_summary, indent=2))
save_json("visual_review.json", {"manifest": manifest, "review": review, "summary": visual_summary})


[
  {
    "method": "bm25",
    "supported": 0,
    "unknown": 0,
    "support_at_5_bounds": [
      0.0,
      0.0
    ]
  },
  {
    "method": "nomic_text",
    "supported": 1,
    "unknown": 0,
    "support_at_5_bounds": [
      0.2,
      0.2
    ]
  },
  {
    "method": "clip_text",
    "supported": 4,
    "unknown": 0,
    "support_at_5_bounds": [
      0.8,
      0.8
    ]
  },
  {
    "method": "rrf_reference",
    "supported": 1,
    "unknown": 0,
    "support_at_5_bounds": [
      0.2,
      0.2
    ]
  }
]
